<a href="https://colab.research.google.com/github/AnnaBastin/Company-WikiBot/blob/main/company.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U langchain langchain-community langchain-text-splitters chromadb
!pip install -q google-generativeai langchain-google-genai
!pip install -q pypdf python-docx docx2txt
!pip install -U langchain-google-genai google-generativeai
!pip install python-dotenv

In [2]:
!git clone https://github.com/AnnaBastin/Company-WikiBot.git

fatal: destination path 'Company-WikiBot' already exists and is not an empty directory.


In [12]:
!ls

chroma_db  Company-WikiBot  documents  sample_data


In [13]:
!ls Company-WikiBot


company.ipynb  documents  README.md


In [14]:
!ls documents

employeebenefits.txt


In [15]:
import os
import glob

from google.colab import files

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    Docx2txtLoader
)

from langchain_community.vectorstores import Chroma

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)

import google.generativeai as genai

In [16]:
from google.colab import userdata
import google.generativeai as genai

api_key = userdata.get("GOOGLE_API_KEY")

genai.configure(api_key=api_key)

print("API configured successfully")

API configured successfully


In [17]:
uploaded = files.upload()

os.makedirs("documents", exist_ok=True)

for filename in uploaded.keys():
    os.rename(filename, f"documents/{filename}")

print(os.listdir("documents"))

Saving employeebenefits.txt to employeebenefits.txt
['employeebenefits.txt']


In [18]:
DOCUMENTS_PATH = "/content/Company-WikiBot/documents"
documents = []

all_files = glob.glob(f"{DOCUMENTS_PATH}/**/*", recursive=True)

print("Files found:")
for f in all_files:
    print(f)

for file_path in all_files:

    if os.path.isdir(file_path):
        continue

    try:
        if file_path.endswith(".txt"):
            loader = TextLoader(file_path)

        elif file_path.endswith(".pdf"):
            loader = PyPDFLoader(file_path)

        elif file_path.endswith(".docx"):
            loader = Docx2txtLoader(file_path)

        else:
            print("Skipping:", file_path)
            continue

        documents.extend(loader.load())
        print("Loaded:", file_path)

    except Exception as e:
        print("Error:", file_path, e)

print("Total docs:", len(documents))

Files found:
/content/Company-WikiBot/documents/placeholdermvp.txt
/content/Company-WikiBot/documents/employeebenefits.txt
/content/Company-WikiBot/documents/fakedoc.txt
/content/Company-WikiBot/documents/falseinfo.txt
/content/Company-WikiBot/documents/leavepolicy.txt
Loaded: /content/Company-WikiBot/documents/placeholdermvp.txt
Loaded: /content/Company-WikiBot/documents/employeebenefits.txt
Loaded: /content/Company-WikiBot/documents/fakedoc.txt
Loaded: /content/Company-WikiBot/documents/falseinfo.txt
Loaded: /content/Company-WikiBot/documents/leavepolicy.txt
Total docs: 5


In [19]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Chunks:", len(chunks))

Chunks: 5


In [20]:
import google.generativeai as genai
import os

#genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
#genai.configure(api_key=api_key)

models = genai.list_models()

for m in models:
    print("MODEL:", m.name)
    print("SUPPORTED METHODS:", m.supported_generation_methods)
    print("-" * 50)

for m in genai.list_models():
    if "embedContent" in m.supported_generation_methods:
        print("EMBEDDING MODEL:", m.name)

MODEL: models/gemini-2.5-flash
SUPPORTED METHODS: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------
MODEL: models/gemini-2.5-pro
SUPPORTED METHODS: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------
MODEL: models/gemini-2.0-flash
SUPPORTED METHODS: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------
MODEL: models/gemini-2.0-flash-001
SUPPORTED METHODS: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------
MODEL: models/gemini-2.0-flash-lite-001
SUPPORTED METHODS: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------
MODEL: models/gemini-2.0-flash-lite
SUPPORTED METHODS: ['generateContent',

In [21]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [22]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

test = embeddings.embed_query("hello world")
print("Embedding length:", len(test))

Embedding length: 3072


In [23]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="/content/chroma_db"
)

retriever = vectorstore.as_retriever()

In [24]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0
)

In [29]:
query = "what is salamandar"

retrieved_docs = retriever.invoke(query)

print("Retrieved docs:", len(retrieved_docs))

if len(retrieved_docs) == 0:
    print("No relevant documents found.")
else:
    context = "\n".join([doc.page_content for doc in retrieved_docs])

    prompt = f"""
    Answer ONLY using the context below.

    Context:
    {context}

    Question:
    {query}
    """

    response = llm.invoke(prompt)

    #output = response.content
    clean_text = response.content[0]["text"]
    print(clean_text)
    #print(output)

Retrieved docs: 4
Based on the provided context, the salamander (or salamandar) is:

* The most important aspect of the company.
* The entity that must review all information.
* The entity that must be contacted for permission before taking leave.
* Something that can be sold to the zoo to be used as monkey fodder, or fed to a bird.
